# Part 4. Writing Sample: Geopolitical Shock Transmission in Emerging Bond Markets
## Evidence from Kazakhstan Following the 2022 Russian Invasion of Ukraine
**Author:** Darkhan Mashirapov  
**Last updated:** 2026-08  
**Purpose:** Estimate ARDL bounds-testing models for Kazakhstan's and Russia's
corporate credit-risk spreads against domestic liquidity and global
macro-financial determinants, testing for a long-run (cointegrating)
relationship and estimating the corresponding error-correction model (ECM).
This addresses H1 (pre-war similarity in spread determinants) directly, and
sets up the baseline against which Notebooks 05 (Markov-switching) and 06
(VAR/IRF) test for post-war structural change.

### Why ARDL bounds testing fits this data

As established in Notebook 03 (Block 8), Kazakhstan's credit spreads are
I(1) while Russia's are I(0) — a mixed order of integration across the two
countries' dependent variables. The ARDL bounds-testing approach (Pesaran,
Shin & Smith, 2001) is specifically designed for this setting: it allows
regressors (and the dependent variable) to be a mix of I(0) and I(1) series
without requiring a common order of integration, provided none are I(2) —
already confirmed and resolved in Notebook 02. This also avoids the need to
pre-test for cointegration using a method (e.g., Engle-Granger, Johansen)
that assumes all series share the same order of integration.

#### Notebook 04 · Block 1 · Load Datasets & Imports

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = "/Users/darhanmashirapov/Desktop/MSF NU/Writing Sample MS2027/1. Data/Results"

levels = pd.read_excel(f"{RESULTS_DIR}/master_dataset_final.xlsx", sheet_name="Levels (Raw)", parse_dates=["date"]).set_index("date")
transformed = pd.read_excel(f"{RESULTS_DIR}/master_dataset_final.xlsx", sheet_name="Transformed (Stationary)", parse_dates=["date"]).set_index("date")
codebook = pd.read_excel(f"{RESULTS_DIR}/master_dataset_final.xlsx", sheet_name="Codebook")

war_onset = pd.Timestamp("2022-02-01")

print(f"Levels shape: {levels.shape}, range {levels.index.min().date()} to {levels.index.max().date()}")
print(f"Transformed shape: {transformed.shape}, range {transformed.index.min().date()} to {transformed.index.max().date()}")

# Check whether the ARDL-specific package is available
try:
    from statsmodels.tsa.ardl import ARDL, ardl_select_order
    print("\n✓ statsmodels ARDL module available")
except ImportError:
    print("\n✗ statsmodels ARDL module NOT available — check statsmodels version (needs >= 0.12)")

Levels shape: (472, 56), range 1987-05-01 to 2026-08-01
Transformed shape: (471, 48), range 1987-06-01 to 2026-08-01

✓ statsmodels ARDL module available


#### Notebook 04 · Block 2 · Define regressor sets and sample splits

In [39]:
# ── Dependent variables: KZ and RU spreads, all three maturities
# Recall from Notebook 03: KZ spreads are I(1) (transformed = differenced),
# RU spreads are I(0) (transformed = levels, "none_needed")
dep_vars_kz = {"1yr": "kz_spread_1y_m", "5yr": "kz_spread_5y_m", "10yr": "kz_spread_10y_m"}
dep_vars_ru = {"1yr": "ru_spread_1y_m", "5yr": "ru_spread_5y_m", "10yr": "ru_spread_10y_m"}

# Candidate regressors, split by category 
# categories table from earlier: domestic liquidity, global factors) 
regressors_kz = {
    "domestic_liquidity": "kz_tonia_m",
    "global_oil": "gl_oil_brent_m",
    "global_vix": "gl_vix_m",
    "us_treasury": None,  # set per-maturity below, to match spread maturity
    "domestic_vol": "kz_vol_m",
}
regressors_ru = {
    "domestic_liquidity": "ru_ruonia_m",
    "global_oil": "gl_oil_brent_m",
    "global_vix": "gl_vix_m",
    "us_treasury": None,  # set per-maturity below
    "domestic_vol": "ru_vol_m",
}

us_treasury_by_maturity = {
    "1yr": "us_yield_1y_m",
    "5yr": "us_yield_5y_m",
    "10yr": "us_yield_10y_m",
}

# Sample splits
pre_war_mask = levels.index < war_onset
full_sample_mask = levels.index == levels.index  # all True, for clarity/symmetry

print("Regressor sets defined:")
print(f"  KZ: {list(regressors_kz.keys())} + maturity-matched US Treasury yield")
print(f"  RU: {list(regressors_ru.keys())} + maturity-matched US Treasury yield")

print(f"\nSample splits:")
print(f"  Pre-war: {pre_war_mask.sum()} obs ({levels.index[pre_war_mask].min().date()} to {levels.index[pre_war_mask].max().date()})")
print(f"  Full sample: {full_sample_mask.sum()} obs ({levels.index.min().date()} to {levels.index.max().date()})")

# Sanity check: confirm all regressor columns actually exist
all_regressor_cols = list(regressors_kz.values()) + list(regressors_ru.values()) + list(us_treasury_by_maturity.values())
all_regressor_cols = [c for c in all_regressor_cols if c is not None]
missing = [c for c in all_regressor_cols if c not in levels.columns]
print(f"\nMissing regressor columns: {missing if missing else 'none, all present'}")

Regressor sets defined:
  KZ: ['domestic_liquidity', 'global_oil', 'global_vix', 'us_treasury', 'domestic_vol'] + maturity-matched US Treasury yield
  RU: ['domestic_liquidity', 'global_oil', 'global_vix', 'us_treasury', 'domestic_vol'] + maturity-matched US Treasury yield

Sample splits:
  Pre-war: 417 obs (1987-05-01 to 2022-01-01)
  Full sample: 472 obs (1987-05-01 to 2026-08-01)

Missing regressor columns: none, all present


#### Notebook 04 · Block 3 · Define three specifications per country

In [40]:
# ── Specification A: Minimal baseline (both countries) ──────────────────────
minimal_regressors_kz = ["kz_tonia_m", "gl_oil_brent_m", "gl_vix_m"]  # + maturity-matched US treasury
minimal_regressors_ru = ["ru_ruonia_m", "gl_oil_brent_m", "gl_vix_m"]  # + maturity-matched US treasury

# ── Specification B: Full model (both countries) ────────────────────────────
full_regressors_kz = [
    "kz_tonia_m", "gl_oil_brent_m", "gl_vix_m",
    "kz_inflation_yoy_m", "kz_gdp_m",
    "kz_extdebt_gdp_ratio", "kz_reserves_m2_ratio",
    "kz_vol_m",
]  # + maturity-matched US treasury

full_regressors_ru = [
    "ru_ruonia_m", "gl_oil_brent_m", "gl_vix_m",
    "ru_inflation_yoy_m", "ru_gdp_m",
    "ru_extdebt_gdp_ratio", "ru_reserves_m2_ratio",
    "ru_vol_m",
]  # + maturity-matched US treasury

specifications = {
    "A_minimal_both":     {"kz": minimal_regressors_kz, "ru": minimal_regressors_ru},
    "B_minimal_kz_full_ru": {"kz": minimal_regressors_kz, "ru": full_regressors_ru},
    "C_full_both":        {"kz": full_regressors_kz, "ru": full_regressors_ru},
}

for spec_name, spec in specifications.items():
    print(f"\n{spec_name}:")
    print(f"  KZ regressors ({len(spec['kz'])}): {spec['kz']}")
    print(f"  RU regressors ({len(spec['ru'])}): {spec['ru']}")

# Sanity check: confirm all columns exist
all_cols_check = set()
for spec in specifications.values():
    all_cols_check.update(spec["kz"])
    all_cols_check.update(spec["ru"])
all_cols_check.update(us_treasury_by_maturity.values())

missing = [c for c in all_cols_check if c not in levels.columns]
print(f"\nMissing columns across all specifications: {missing if missing else 'none — all present'}")

# bservation count per regressor, to flag power concerns before fitting
print(f"\n{'='*60}\nParameter-count sanity check (pre-war KZ sample = 27 obs)\n{'='*60}")
for spec_name, spec in specifications.items():
    n_kz_regressors = len(spec["kz"]) + 1  # +1 for maturity-matched US treasury
    print(f"  {spec_name}: {n_kz_regressors} KZ regressors -> "
          f"even 1 lag each needs ~{n_kz_regressors*2} parameters (incl. own-lag), "
          f"against 27 pre-war obs")


A_minimal_both:
  KZ regressors (3): ['kz_tonia_m', 'gl_oil_brent_m', 'gl_vix_m']
  RU regressors (3): ['ru_ruonia_m', 'gl_oil_brent_m', 'gl_vix_m']

B_minimal_kz_full_ru:
  KZ regressors (3): ['kz_tonia_m', 'gl_oil_brent_m', 'gl_vix_m']
  RU regressors (8): ['ru_ruonia_m', 'gl_oil_brent_m', 'gl_vix_m', 'ru_inflation_yoy_m', 'ru_gdp_m', 'ru_extdebt_gdp_ratio', 'ru_reserves_m2_ratio', 'ru_vol_m']

C_full_both:
  KZ regressors (8): ['kz_tonia_m', 'gl_oil_brent_m', 'gl_vix_m', 'kz_inflation_yoy_m', 'kz_gdp_m', 'kz_extdebt_gdp_ratio', 'kz_reserves_m2_ratio', 'kz_vol_m']
  RU regressors (8): ['ru_ruonia_m', 'gl_oil_brent_m', 'gl_vix_m', 'ru_inflation_yoy_m', 'ru_gdp_m', 'ru_extdebt_gdp_ratio', 'ru_reserves_m2_ratio', 'ru_vol_m']

Missing columns across all specifications: none — all present

Parameter-count sanity check (pre-war KZ sample = 27 obs)
  A_minimal_both: 4 KZ regressors -> even 1 lag each needs ~8 parameters (incl. own-lag), against 27 pre-war obs
  B_minimal_kz_full_ru: 4 KZ r

#### Notebook 04 · Block 4 · Fit first ARDL model (Manually enforced lag structure) — KZ 1yr, Specification A, Pre-war

In [41]:
from statsmodels.tsa.ardl import ARDL, UECM

dep_var = "kz_spread_1y_m"
regressors = minimal_regressors_kz  # drop us_yield_1y_m for this run — selection excluded it anyway
model_df = levels.loc[pre_war_mask, [dep_var] + regressors].dropna()

print(f"Modeling sample: {model_df.shape[0]} observations")

# ── Manually force every regressor to have at least 1 lag, since the bounds
# test (UECM) requires this. Own-lag (AR order) = 1, exog order = 1 for all. ──
manual_order = {var: 1 for var in regressors}

ardl_model = ARDL(
    model_df[dep_var],
    lags=1,
    exog=model_df[regressors],
    order=manual_order,
    trend="c",
)
ardl_fit = ardl_model.fit()
print(ardl_fit.summary())

Modeling sample: 27 observations
                              ARDL Model Results                              
Dep. Variable:         kz_spread_1y_m   No. Observations:                   27
Model:               ARDL(1, 1, 1, 1)   Log Likelihood                  10.932
Method:               Conditional MLE   S.D. of innovations              0.159
Date:                Fri, 04 Sep 2026   AIC                             -3.864
Time:                        10:33:03   BIC                              7.459
Sample:                    12-01-2019   HQIC                            -0.603
                         - 01-01-2022                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 0.1763      0.745      0.237      0.816      -1.390       1.743
kz_spread_1y_m.L1     0.9278      0.124      7.480      0.000       0.667    

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


#### Note: Block 4 — first ARDL model, KZ 1yr, minimal spec, pre-war

Model converged with ARDL(1,1,1,0), 7 parameters against 27 observations.
US Treasury 1yr yield was dropped by automatic lag-order selection likely
reflecting collinearity with VIX/oil given the small sample, rather than a
substantive finding that US yields don't matter. This should be noted as a
data-constraint limitation.

The spread's own-lag coefficient (0.93, p<0.001) confirms high persistence,
consistent with the I(1) behavior established in Notebook 03. TONIA shows a
short-run overshoot-then-partial-reversal pattern (contemporaneous +0.35,
1-lag -0.20, both significant). Oil's 1-lag effect is negative and borderline
significant, consistent with the oil-exporter credit-risk story from the EDA.
VIX's negative contemporaneous coefficient is counterintuitive relative to
conventional risk-aversion pricing and should be treated cautiously given the
small, COVID-period-dominated pre-war sample.

**Next step:** run the bounds test for cointegration to determine whether a
genuine long-run relationship exists among these variables, before
interpreting these short-run coefficients as economically meaningful.

#### Notebook 04 · Block 5 · Bounds test for conintegration — KZ 1yr, Specification A, Pre-war

In [42]:
# Convert the selected ARDL model into its equivalent Unrestricted Error
# Correction form — this is the representation the bounds test is run on
uecm_model = UECM.from_ardl(ardl_model)
uecm_fit = uecm_model.fit()
print(uecm_fit.summary())

bounds_test = uecm_fit.bounds_test(case=3)
print(f"\n{'='*60}\nBounds Test for Cointegration (Case 3: unrestricted constant)\n{'='*60}")
print(bounds_test)

                              UECM Model Results                              
Dep. Variable:       D.kz_spread_1y_m   No. Observations:                   27
Model:               UECM(1, 1, 1, 1)   Log Likelihood                  10.932
Method:               Conditional MLE   S.D. of innovations              1.188
Date:                Fri, 04 Sep 2026   AIC                             -3.864
Time:                        10:33:06   BIC                              7.459
Sample:                    12-01-2019   HQIC                            -0.603
                         - 01-01-2022                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                   0.1763      0.745      0.237      0.816      -1.390       1.743
kz_spread_1y_m.L1      -0.0722      0.124     -0.582      0.568      -0.333       0.188
kz_tonia_m.L1   

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


#### Note: Block 5 — Bounds test result, KZ 1yr, minimal spec, pre-war

**Result:** F-statistic = 2.168; upper bound p-value = 0.462, lower bound
p-value = 0.156 (Case 3: unrestricted constant, no trend). Both p-values
exceed conventional significance thresholds, so we fail to reject the null of
no cointegration — no statistically detectable long-run relationship between
Kazakhstan's 1yr credit spread and domestic liquidity, oil, and VIX in the
pre-war period.

**Interpretation:** given the error-correction coefficient itself was also
insignificant (kz_spread_1y_m.L1 = -0.072, p=0.568), this null result is
consistent across both diagnostics. However, this should be interpreted
primarily as a **power limitation from the small pre-war sample (27
observations)**, not as strong evidence that no true long-run relationship
exists. Testing the same specification on Russia's much longer pre-war sample
(109 observations) will help clarify whether this is a data constraint
specific to Kazakhstan or a genuine substantive finding.

**Implication for H1:** the inability to establish cointegration for
Kazakhstan's pre-war spread should be documented as a data limitation
affecting the strength of any claim about pre-war similarity between the two
countries, rather than interpreted as evidence against H1 outright.

#### Notebook 04 · Block 6 · Fit ARDL + Bounds Test — RU 1yr, Specification A, Pre-war

In [30]:
dep_var_ru = "ru_spread_1y_m"
regressors_ru_a = minimal_regressors_ru  # ["ru_ruonia_m", "gl_oil_brent_m", "gl_vix_m"]

model_df_ru = levels.loc[pre_war_mask, [dep_var_ru] + regressors_ru_a].dropna()
print(f"Modeling sample: {model_df_ru.shape[0]} observations")
print(f"Date range: {model_df_ru.index.min().date()} to {model_df_ru.index.max().date()}")

# ── Same manual lag enforcement as KZ: every regressor gets at least 1 lag,
# required for the bounds test (UECM) to work ────────────────────────────────
manual_order_ru = {var: 1 for var in regressors_ru_a}

ardl_model_ru = ARDL(
    model_df_ru[dep_var_ru],
    lags=1,
    exog=model_df_ru[regressors_ru_a],
    order=manual_order_ru,
    trend="c",
)
ardl_fit_ru = ardl_model_ru.fit()
print(ardl_fit_ru.summary())

# ── Bounds test ───────────────────────────────────────────────────────────────
uecm_model_ru = UECM.from_ardl(ardl_model_ru)
uecm_fit_ru = uecm_model_ru.fit()
print(uecm_fit_ru.summary())

bounds_test_ru = uecm_fit_ru.bounds_test(case=3)
print(f"\n{'='*60}\nBounds Test for Cointegration — RU 1yr, Pre-War (Case 3)\n{'='*60}")
print(bounds_test_ru)

Modeling sample: 109 observations
Date range: 2013-01-01 to 2022-01-01
                              ARDL Model Results                              
Dep. Variable:         ru_spread_1y_m   No. Observations:                  109
Model:               ARDL(1, 1, 1, 1)   Log Likelihood                 -27.367
Method:               Conditional MLE   S.D. of innovations              0.312
Date:                Wed, 02 Sep 2026   AIC                             72.733
Time:                        18:44:17   BIC                             96.872
Sample:                    02-01-2013   HQIC                            82.521
                         - 01-01-2022                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.5155      0.302     -1.707      0.091      -1.115       0.084
ru_spread_1y_m.L1     0.6824      0.067

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


#### Note: Block 6 — Bounds test result, RU 1yr, minimal spec, pre-war

**Result:** F-statistic = 6.33; both bound p-values far below 0.05 (upper
p=0.0012, lower p<0.0001) — clear rejection of no-cointegration. Russia's 1yr
spread shows a statistically robust long-run relationship with RUONIA, oil,
and VIX pre-war.

**Error-correction speed:** coefficient -0.318 (p<0.001), implying ~32% of any
deviation from equilibrium is corrected monthly (half-life ≈ 1.8 months) —
the textbook signature of a well-identified cointegrating relationship.

**Cross-country contrast with KZ (Block 5):** RU's result is a clean,
confident cointegration finding; KZ's was a clean non-rejection, very likely
reflecting insufficient statistical power from its 27-observation pre-war
sample rather than a genuine absence of relationship. Notably, even where KZ
produced point estimates, several long-run signs differ from RU's (oil:
negative for KZ vs. positive for RU; VIX: negative for KZ vs. positive,
conventional-sign for RU) — suggestive of different underlying dynamics, but
this should be stated as a tentative observation given KZ's data limitation,
not a confirmed cross-country difference.

**Implication for H1:** Russia's result supports H1's premise that pre-war
credit spreads were driven by an identifiable set of common macro-financial
determinants. Kazakhstan's inconclusive result means H1 cannot be tested with
equal confidence for both countries using this specification — this should be
explicitly acknowledged as a limitation, with the RU result treated as the
stronger evidence base pending further robustness checks (e.g., Specification
B/C, or alternative small-sample methods for KZ).

#### Notebook 04 · Block 7 · Reusable ARDL + Bounds Test Function

In [31]:
def run_ardl_bounds_test(dep_var, regressors, sample_mask, sample_label, lag=1, case=3):
    model_df = levels.loc[sample_mask, [dep_var] + regressors].dropna()
    n_obs = model_df.shape[0]

    print(f"\n{'='*70}\n  {dep_var} | {sample_label} | n={n_obs}\n{'='*70}")

    manual_order = {var: lag for var in regressors}

    ardl_model = ARDL(
        model_df[dep_var], lags=lag,
        exog=model_df[regressors], order=manual_order,
        trend="c",
    )
    ardl_fit = ardl_model.fit()

    uecm_model = UECM.from_ardl(ardl_model)
    uecm_fit = uecm_model.fit()

    bounds_test = uecm_fit.bounds_test(case=case)

    ec_coef = uecm_fit.params.get(f"{dep_var}.L1", np.nan)
    ec_pval = uecm_fit.pvalues.get(f"{dep_var}.L1", np.nan)

    # p_values is a namedtuple-like structure; access by field name for safety
    lower_p = bounds_test.p_values.lower
    upper_p = bounds_test.p_values.upper

    print(f"Error-correction term: {ec_coef:.4f} (p={ec_pval:.4f})")
    print(f"Bounds test F-stat: {bounds_test.stat:.4f} | "
          f"upper p={upper_p:.4f} | lower p={lower_p:.4f}")

    if upper_p < 0.05:
        verdict = "COINTEGRATION (reject null at both bounds)"
    elif lower_p < 0.05:
        verdict = "INCONCLUSIVE (reject lower bound only)"
    else:
        verdict = "NO COINTEGRATION (fail to reject)"
    print(f"Verdict: {verdict}")

    return dict(ardl_fit=ardl_fit, uecm_fit=uecm_fit, bounds_test=bounds_test,
                n_obs=n_obs, ec_coef=ec_coef, ec_pval=ec_pval, verdict=verdict)


# ── Run Specification A across remaining maturities: KZ 5yr, KZ 10yr, RU 5yr, RU 10yr ──
results_specA = {}

results_specA["kz_5yr"] = run_ardl_bounds_test(
    "kz_spread_5y_m", minimal_regressors_kz, pre_war_mask, "Pre-War"
)
results_specA["kz_10yr"] = run_ardl_bounds_test(
    "kz_spread_10y_m", minimal_regressors_kz, pre_war_mask, "Pre-War"
)
results_specA["ru_5yr"] = run_ardl_bounds_test(
    "ru_spread_5y_m", minimal_regressors_ru, pre_war_mask, "Pre-War"
)
results_specA["ru_10yr"] = run_ardl_bounds_test(
    "ru_spread_10y_m", minimal_regressors_ru, pre_war_mask, "Pre-War"
)


  kz_spread_5y_m | Pre-War | n=27
Error-correction term: -0.2652 (p=0.0684)
Bounds test F-stat: 1.1122 | upper p=0.8837 | lower p=0.5950
Verdict: NO COINTEGRATION (fail to reject)

  kz_spread_10y_m | Pre-War | n=27
Error-correction term: -0.3019 (p=0.0238)
Bounds test F-stat: 1.9861 | upper p=0.5380 | lower p=0.2038
Verdict: NO COINTEGRATION (fail to reject)

  ru_spread_5y_m | Pre-War | n=109
Error-correction term: -0.3236 (p=0.0000)
Bounds test F-stat: 8.7810 | upper p=0.0000 | lower p=0.0000
Verdict: COINTEGRATION (reject null at both bounds)

  ru_spread_10y_m | Pre-War | n=109
Error-correction term: -0.3285 (p=0.0000)
Bounds test F-stat: 7.6345 | upper p=0.0001 | lower p=0.0000
Verdict: COINTEGRATION (reject null at both bounds)


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

#### Note: Block 7 — Specification A results across all maturities, pre-war

**Full results:**

| Country | Maturity | n | EC term (p-value) | Bounds F-stat | Verdict |
|---|---|---|---|---|---|
| KZ | 1yr | 27 | -0.072 (0.568) | 2.17 | No cointegration |
| KZ | 5yr | 27 | -0.265 (0.068) | 1.11 | No cointegration |
| KZ | 10yr | 27 | -0.302 (0.024) | 1.99 | No cointegration |
| RU | 1yr | 109 | -0.318 (<0.001) | 6.33 | Cointegration |
| RU | 5yr | 109 | -0.328 (<0.001) | 8.78 | Cointegration |
| RU | 10yr | 109 | -0.328 (<0.001) | 7.63 | Cointegration |

**Consistent country-level split, not maturity-driven:** all three Russian
maturities show strong, statistically confident cointegration (bounds F-stats
6.3–8.8, error-correction coefficients tightly clustered around -0.32 to
-0.33 and highly significant, p<0.001). All three Kazakhstan maturities fail
to reject the null of no cointegration under the joint bounds test.

**A subtlety worth noting:** Kazakhstan's individual error-correction
coefficients approach or reach conventional significance as maturity
increases (10yr: p=0.024, technically significant on its own), yet the joint
bounds F-test still indicates no cointegration even at 10yr. This is not a
contradiction — the bounds test is a joint test across all lagged-level
terms, and is the authoritative verdict in the Pesaran-Shin-Smith framework;
a single coefficient's own significance does not override it. This
distinction should be stated explicitly in the methodology section to avoid
the appearance of cherry-picking whichever statistic looks more favorable.

**Cross-country implication for H1:** given this pattern holds uniformly
across all three maturities (not just 1yr), the case that Kazakhstan's
non-result is a **small-sample power issue rather than a genuine absence of
long-run relationship** is strengthened — a real economic difference in
underlying dynamics would more plausibly show maturity-dependent variation,
whereas a uniform "fail everywhere" pattern for KZ alongside a uniform "pass
everywhere" pattern for RU is exactly what you'd expect if the deciding
factor is sample size (27 vs. 109) rather than an actual structural
difference between the two countries' pre-war credit markets.

#### Notebook 04 · Block 8 · Specification B — Minimal KZ, Full RU, Pre-war

In [32]:
# KZ: same minimal regressors as Specification A (no change)
# RU: full regressor set (adds inflation, GDP, extdebt/GDP, reserves/M2, vol)

results_specB = {}

for maturity, kz_col, ru_col in [("1yr", "kz_spread_1y_m", "ru_spread_1y_m"),
                                   ("5yr", "kz_spread_5y_m", "ru_spread_5y_m"),
                                   ("10yr", "kz_spread_10y_m", "ru_spread_10y_m")]:

    # KZ: identical to Spec A, included here for direct side-by-side comparison
    results_specB[f"kz_{maturity}"] = run_ardl_bounds_test(
        kz_col, minimal_regressors_kz, pre_war_mask, "Pre-War"
    )

    # RU: full regressor set
    results_specB[f"ru_{maturity}"] = run_ardl_bounds_test(
        ru_col, full_regressors_ru, pre_war_mask, "Pre-War"
    )


  kz_spread_1y_m | Pre-War | n=27
Error-correction term: -0.0722 (p=0.5678)
Bounds test F-stat: 2.1680 | upper p=0.4625 | lower p=0.1564
Verdict: NO COINTEGRATION (fail to reject)

  ru_spread_1y_m | Pre-War | n=74
Error-correction term: -0.5030 (p=0.0000)
Bounds test F-stat: 5.3060 | upper p=0.0002 | lower p=0.0000
Verdict: COINTEGRATION (reject null at both bounds)

  kz_spread_5y_m | Pre-War | n=27
Error-correction term: -0.2652 (p=0.0684)
Bounds test F-stat: 1.1122 | upper p=0.8837 | lower p=0.5950
Verdict: NO COINTEGRATION (fail to reject)

  ru_spread_5y_m | Pre-War | n=74
Error-correction term: -0.4178 (p=0.0001)
Bounds test F-stat: 2.8289 | upper p=0.1401 | lower p=0.0059
Verdict: INCONCLUSIVE (reject lower bound only)

  kz_spread_10y_m | Pre-War | n=27
Error-correction term: -0.3019 (p=0.0238)
Bounds test F-stat: 1.9861 | upper p=0.5380 | lower p=0.2038
Verdict: NO COINTEGRATION (fail to reject)

  ru_spread_10y_m | Pre-War | n=74
Error-correction term: -0.3057 (p=0.0075)
Bo

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

#### Note: Block 8 — Specification B results, pre-war

**Full results:**

| Country | Maturity | n | EC term (p-value) | Bounds F-stat | Verdict |
|---|---|---|---|---|---|
| KZ | 1yr | 27 | -0.072 (0.568) | 2.17 | No cointegration |
| KZ | 5yr | 27 | -0.265 (0.068) | 1.11 | No cointegration |
| KZ | 10yr | 27 | -0.302 (0.024) | 1.99 | No cointegration |
| RU | 1yr | 74 | -0.503 (<0.001) | 5.31 | Cointegration |
| RU | 5yr | 74 | -0.418 (<0.001) | 2.83 | Inconclusive |
| RU | 10yr | 74 | -0.306 (0.008) | 1.51 | No cointegration |

**Sample size cost:** adding the full regressor set (inflation, GDP,
extdebt/GDP, reserves/M2, volatility) reduces RU's usable pre-war sample from
109 to 74 observations, since several of these variables (particularly the
annualized fiscal ratios) have shorter coverage than RUONIA/oil/VIX alone.

**Key finding — RU's cointegration result is maturity-sensitive to
specification:** under the minimal specification (Block 7), RU showed strong
cointegration at all three maturities. Under the full specification, this
holds cleanly only at 1yr; 5yr weakens to inconclusive, and 10yr weakens all
the way to no cointegration — the same verdict as Kazakhstan. This is
consistent with the smaller sample and larger parameter count compounding the
same statistical-power problem already documented for Kazakhstan throughout
this notebook, now appearing in Russia's results once its effective sample
size and model complexity move closer to Kazakhstan's constrained regime.

**Implication for the paper:** the strength of RU's pre-war cointegration
finding should be reported as specification-dependent, not as a uniformly
robust result across all model choices. The minimal specification (Spec A)
provides the most confident evidence for RU across all maturities; the full
specification's weaker results at longer maturities should be disclosed
directly rather than omitted, since selectively reporting only the favorable
specification would misrepresent the robustness of the finding.

#### Notebook 04 · Block 9 · Specification C — Full Model, both countries, PRE-WAR

In [33]:
results_specC = {}

for maturity, kz_col, ru_col in [("1yr", "kz_spread_1y_m", "ru_spread_1y_m"),
                                   ("5yr", "kz_spread_5y_m", "ru_spread_5y_m"),
                                   ("10yr", "kz_spread_10y_m", "ru_spread_10y_m")]:

    print(f"\n{'*'*70}\n  MATURITY: {maturity}\n{'*'*70}")

    try:
        results_specC[f"kz_{maturity}"] = run_ardl_bounds_test(
            kz_col, full_regressors_kz, pre_war_mask, "Pre-War"
        )
    except Exception as e:
        print(f"\n✗ KZ {maturity} FAILED: {type(e).__name__}: {e}")
        results_specC[f"kz_{maturity}"] = None

    try:
        results_specC[f"ru_{maturity}"] = run_ardl_bounds_test(
            ru_col, full_regressors_ru, pre_war_mask, "Pre-War"
        )
    except Exception as e:
        print(f"\n✗ RU {maturity} FAILED: {type(e).__name__}: {e}")
        results_specC[f"ru_{maturity}"] = None


**********************************************************************
  MATURITY: 1yr
**********************************************************************

  kz_spread_1y_m | Pre-War | n=27
Error-correction term: -0.6689 (p=0.1492)
Bounds test F-stat: 1.1791 | upper p=0.9349 | lower p=0.4690
Verdict: NO COINTEGRATION (fail to reject)

  ru_spread_1y_m | Pre-War | n=74
Error-correction term: -0.5030 (p=0.0000)
Bounds test F-stat: 5.3060 | upper p=0.0002 | lower p=0.0000
Verdict: COINTEGRATION (reject null at both bounds)

**********************************************************************
  MATURITY: 5yr
**********************************************************************

  kz_spread_5y_m | Pre-War | n=27
Error-correction term: -0.0914 (p=0.8035)
Bounds test F-stat: 1.4996 | upper p=0.8135 | lower p=0.2508
Verdict: NO COINTEGRATION (fail to reject)

  ru_spread_5y_m | Pre-War | n=74
Error-correction term: -0.4178 (p=0.0001)
Bounds test F-stat: 2.8289 | upper p=0.1401 | lower p

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

#### Note: Block 9 — Specification C results, pre-war

**Full results:**

| Country | Maturity | n | EC term (p-value) | Bounds F-stat | Verdict |
|---|---|---|---|---|---|
| KZ | 1yr | 27 | -0.669 (0.149) | 1.18 | No cointegration |
| KZ | 5yr | 27 | -0.091 (0.804) | 1.50 | No cointegration |    
| KZ | 10yr | 27 | -0.018 (0.941) | 2.22 | Inconclusive |
| RU | 1yr | 74 | -0.503 (<0.001) | 5.31 | Cointegration |
| RU | 5yr | 74 | -0.418 (<0.001) | 2.83 | Inconclusive |
| RU | 10yr | 74 | -0.306 (0.008) | 1.51 | No cointegration |


**RU results are identical to Specification B** (Block 8), since RU's full
regressor set was already used there — no new information for Russia beyond
confirming the same specification-sensitivity already documented.

**KZ's full specification did not fail outright, but its estimates become
progressively less meaningful as maturity increases.** The error-correction
coefficient's p-value rises from 0.149 (1yr) to 0.804 (5yr) to 0.941 (10yr) —
by 10yr, the coefficient carries essentially no statistical information. This
is the empirical realization of the overfitting concern raised in Block 2:
with 8 regressors and enforced lags estimated on only 27 observations, the
model technically converges but produces estimates too imprecise to
interpret, rather than genuine evidence of no relationship.

**Overall three-specification synthesis:** Kazakhstan fails to show
cointegration under every specification tested, with results becoming less
reliable (not more informative) as more regressors are added given its fixed
27-observation constraint. Russia shows robust cointegration only under the
minimal specification across all maturities; its result weakens
progressively at longer maturities once the full regressor set is
introduced, driven by both a smaller effective sample (74 vs. 109) and a
higher parameter count. Taken together, these results support reporting
Specification A (minimal) as the primary result for both countries, with
Specifications B and C serving as documented robustness/limitation checks
rather than alternative preferred models.

#### Notebook 04 · Block 10 · Specification A, Full Sample with war dummy

In [34]:
# Add war_dummy to the regressor set for the full-sample models
minimal_regressors_kz_dummy = minimal_regressors_kz + ["war_dummy"]
minimal_regressors_ru_dummy = minimal_regressors_ru + ["war_dummy"]

results_specA_fullsample = {}

for maturity, kz_col, ru_col in [("1yr", "kz_spread_1y_m", "ru_spread_1y_m"),
                                   ("5yr", "kz_spread_5y_m", "ru_spread_5y_m"),
                                   ("10yr", "kz_spread_10y_m", "ru_spread_10y_m")]:

    print(f"\n{'*'*70}\n  MATURITY: {maturity}\n{'*'*70}")

    results_specA_fullsample[f"kz_{maturity}"] = run_ardl_bounds_test(
        kz_col, minimal_regressors_kz_dummy, full_sample_mask, "Full Sample + War Dummy"
    )
    results_specA_fullsample[f"ru_{maturity}"] = run_ardl_bounds_test(
        ru_col, minimal_regressors_ru_dummy, full_sample_mask, "Full Sample + War Dummy"
    )


**********************************************************************
  MATURITY: 1yr
**********************************************************************

  kz_spread_1y_m | Full Sample + War Dummy | n=82
Error-correction term: -0.2002 (p=0.0005)
Bounds test F-stat: 3.9584 | upper p=0.0377 | lower p=0.0031
Verdict: COINTEGRATION (reject null at both bounds)

  ru_spread_1y_m | Full Sample + War Dummy | n=164
Error-correction term: -0.3335 (p=0.0000)
Bounds test F-stat: 7.9568 | upper p=0.0000 | lower p=0.0000
Verdict: COINTEGRATION (reject null at both bounds)

**********************************************************************
  MATURITY: 5yr
**********************************************************************

  kz_spread_5y_m | Full Sample + War Dummy | n=82
Error-correction term: -0.1152 (p=0.0392)
Bounds test F-stat: 1.4946 | upper p=0.7655 | lower p=0.3559
Verdict: NO COINTEGRATION (fail to reject)

  ru_spread_5y_m | Full Sample + War Dummy | n=164
Error-correction ter

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

#### Note: Block 9 - Specification A, full sample with war dummy

**Full results:**

| Country | Maturity | n | EC term (p-value) | Bounds F-stat | Verdict |
|---|---|---|---|---|---|
| KZ | 1yr | 82 | -0.200 (<0.001) | 3.96 | Cointegration |
| KZ | 5yr | 82 | -0.115 (0.039) | 1.49 | No cointegration |
| KZ | 10yr | 82 | -0.149 (0.010) | 2.37 | No cointegration |
| RU | 1yr | 164 | -0.334 (<0.001) | 7.96 | Cointegration |
| RU | 5yr | 164 | -0.201 (<0.001) | 3.99 | Cointegration |
| RU | 10yr | 164 | -0.164 (<0.001) | 2.94 | Inconclusive |

**Key finding - Kazakhstan's 1yr spread now shows confirmed cointegration**,
in contrast to every pre-war-only specification (Blocks 6, 7, 8), where it
consistently failed. With the sample roughly tripling (27 → 82 observations)
by including the post-war period, this directly supports the paper's running
interpretation that Kazakhstan's earlier null results reflected insufficient
statistical power rather than a genuine absence of long-run relationship.

**Maturity-dependent pattern persists:** Kazakhstan's 5yr and 10yr spreads
still fail to show cointegration even with the larger full sample, echoing
the term-structure heterogeneity first identified in the EDA, where oil-decoupling and domestic-liquidity effects were
found to strengthen with maturity. This suggests the minimal regressor set
(TONIA, oil, VIX, war dummy) captures the long-run dynamics of Kazakhstan's
short-term spread well, but not its longer-maturity spreads — a fuller
specification or a dummy-interaction term may be needed for 5yr/10yr.

**Russia shows a parallel, softer weakening pattern** by maturity (1yr and
5yr confirmed cointegration; 10yr only inconclusive), consistent with the
specification-sensitivity already documented in Blocks 8–9.

**Implication for the paper's narrative:** this result meaningfully
strengthens the case that Kazakhstan's credit spread — at least at the short
end — is genuinely linked to the same macro-financial determinants as
Russia's, once the war's level-shift effect is properly accounted for via the
dummy. The persistent 5yr/10yr non-result for Kazakhstan should be reported
honestly as a maturity-specific limitation, directly connecting back to the
EDA's finding that Kazakhstan's long-end response to the war differs
structurally from its short end.

#### Notebook 04 · Block 10 · Expanded post-war specification — KZ 5yr, 10yr

In [35]:
# Adding fiscal/external variables one at a time to KZ's minimal spec,
# to see if any recover cointegration at 5yr/10yr where TONIA/oil/VIX alone
# failed. Testing one variable at a time (rather than the full Spec C set)
# avoids repeating the overfitting problem from Block 9.

candidate_additions = {
    "extdebt_gdp": "kz_extdebt_gdp_ratio",
    "reserves_m2": "kz_reserves_m2_ratio",
    "inflation": "kz_inflation_yoy_m",
    "gdp": "kz_gdp_m",
}

results_kz_expanded = {}

for maturity, dep_col in [("5yr", "kz_spread_5y_m"), ("10yr", "kz_spread_10y_m")]:
    print(f"\n{'*'*70}\n  KZ {maturity} — testing one additional variable at a time\n{'*'*70}")

    for label, extra_var in candidate_additions.items():
        regressors_test = minimal_regressors_kz_dummy + [extra_var]
        print(f"\n--- Adding: {label} ({extra_var}) ---")
        try:
            results_kz_expanded[f"{maturity}_{label}"] = run_ardl_bounds_test(
                dep_col, regressors_test, full_sample_mask, "Full Sample + War Dummy"
            )
        except Exception as e:
            print(f"✗ FAILED: {type(e).__name__}: {e}")
            results_kz_expanded[f"{maturity}_{label}"] = None


**********************************************************************
  KZ 5yr — testing one additional variable at a time
**********************************************************************

--- Adding: extdebt_gdp (kz_extdebt_gdp_ratio) ---

  kz_spread_5y_m | Full Sample + War Dummy | n=77
Error-correction term: -0.1077 (p=0.0787)
Bounds test F-stat: 1.2182 | upper p=0.8875 | lower p=0.4890
Verdict: NO COINTEGRATION (fail to reject)

--- Adding: reserves_m2 (kz_reserves_m2_ratio) ---

  kz_spread_5y_m | Full Sample + War Dummy | n=80
Error-correction term: -0.1368 (p=0.0197)
Bounds test F-stat: 1.3007 | upper p=0.8592 | lower p=0.4362
Verdict: NO COINTEGRATION (fail to reject)

--- Adding: inflation (kz_inflation_yoy_m) ---

  kz_spread_5y_m | Full Sample + War Dummy | n=81
Error-correction term: -0.1270 (p=0.0734)
Bounds test F-stat: 1.5990 | upper p=0.7313 | lower p=0.2733
Verdict: NO COINTEGRATION (fail to reject)

--- Adding: gdp (kz_gdp_m) ---

  kz_spread_5y_m | Full Samp

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

#### Note: Block 10 — Expanded specification test, KZ 5yr/10yr

**Full results:**

| Maturity | Added variable | n | EC (p-value) | F-stat | Verdict |
|---|---|---|---|---|---|
| 5yr | extdebt/GDP | 77 | -0.108 (0.079) | 1.22 | No cointegration |
| 5yr | reserves/M2 | 80 | -0.137 (0.020) | 1.30 | No cointegration |
| 5yr | inflation | 81 | -0.127 (0.073) | 1.60 | No cointegration |
| 5yr | GDP | 77 | -0.088 (0.144) | 1.37 | No cointegration |
| 10yr | extdebt/GDP | 77 | -0.171 (0.006) | 2.33 | No cointegration |
| 10yr | reserves/M2 | 80 | -0.165 (0.020) | 2.05 | No cointegration |
| 10yr | inflation | 81 | -0.132 (0.047) | 1.64 | No cointegration |
| 10yr | GDP | 77 | -0.129 (0.042) | 2.11 | No cointegration |

**None of the four candidate variables recovered cointegration** at either
maturity, even though several individual error-correction coefficients reach
conventional significance on their own (e.g., 10yr + extdebt/GDP, p=0.006) —
the joint bounds F-test remains decisively below the lower critical bound in
every case, consistent with the coefficient-vs-joint-test distinction
documented in Block 6.

**Conclusion:** this is treated as a genuine, informative null result rather
than an unresolved gap. Kazakhstan's 5yr and 10yr spreads do not exhibit a
detectable long-run relationship with domestic liquidity, global risk
factors, or the fiscal/macro variables tested here — even after accounting
for the war via the level-shift dummy. Combined with the EDA finding that
oil-decoupling and liquidity-sensitivity effects were strongest specifically
at the long end, this suggests Kazakhstan's long-maturity spread may be
driven by maturity-specific factors (market liquidity, specific issuance
dynamics, investor composition) not captured by the macro fundamentals
examined in this paper — a legitimate scope limitation to state directly
rather than a modeling failure to keep pursuing.

#### Notebook 04 · Block 11 · Full interpretation — RU (PRE-WAR) AND KZ (FULL SAMPLE+DUMMY)

In [36]:
# ── RU: Spec A, pre-war, all 3 maturities ────────────────────────────────────
ru_specA_full = {}
for maturity, dep_col in [("1yr", "ru_spread_1y_m"), ("5yr", "ru_spread_5y_m"), ("10yr", "ru_spread_10y_m")]:
    model_df = levels.loc[pre_war_mask, [dep_col] + minimal_regressors_ru].dropna()
    manual_order = {var: 1 for var in minimal_regressors_ru}
    ardl_model = ARDL(model_df[dep_col], lags=1, exog=model_df[minimal_regressors_ru],
                       order=manual_order, trend="c")
    uecm_fit = UECM.from_ardl(ardl_model).fit()
    ru_specA_full[maturity] = uecm_fit
    print(f"\n{'='*70}\n  RU {maturity} (Pre-War, Spec A) — UECM Results\n{'='*70}")
    print(uecm_fit.summary())

# ── KZ: Spec A + war dummy, full sample, 1yr only (the one that achieved
# cointegration — 5yr/10yr did not, per Block 10) ────────────────────────────
kz_specA_dummy_full = {}
dep_col = "kz_spread_1y_m"
model_df = levels.loc[full_sample_mask, [dep_col] + minimal_regressors_kz_dummy].dropna()
manual_order = {var: 1 for var in minimal_regressors_kz_dummy}
ardl_model = ARDL(model_df[dep_col], lags=1, exog=model_df[minimal_regressors_kz_dummy],
                   order=manual_order, trend="c")
uecm_fit = UECM.from_ardl(ardl_model).fit()
kz_specA_dummy_full["1yr"] = uecm_fit
print(f"\n{'='*70}\n  KZ 1yr (Full Sample + War Dummy, Spec A) — UECM Results\n{'='*70}")
print(uecm_fit.summary())


  RU 1yr (Pre-War, Spec A) — UECM Results
                              UECM Model Results                              
Dep. Variable:       D.ru_spread_1y_m   No. Observations:                  109
Model:               UECM(1, 1, 1, 1)   Log Likelihood                 -27.367
Method:               Conditional MLE   S.D. of innovations              1.596
Date:                Wed, 02 Sep 2026   AIC                             72.733
Time:                        18:44:19   BIC                             96.872
Sample:                    02-01-2013   HQIC                            82.521
                         - 01-01-2022                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.5155      0.302     -1.707      0.091      -1.115       0.084
ru_spread_1y_m.L1      -0.3176      0.067     -4.774      0.0

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

#### Notebook 04 · Block 12 · RU, Full sample + war dummy (for fair comparison with KZ)

In [37]:
ru_specA_dummy_full = {}
for maturity, dep_col in [("1yr", "ru_spread_1y_m"), ("5yr", "ru_spread_5y_m"), ("10yr", "ru_spread_10y_m")]:
    model_df = levels.loc[full_sample_mask, [dep_col] + minimal_regressors_ru_dummy].dropna()
    manual_order = {var: 1 for var in minimal_regressors_ru_dummy}
    ardl_model = ARDL(model_df[dep_col], lags=1, exog=model_df[minimal_regressors_ru_dummy],
                       order=manual_order, trend="c")
    uecm_fit = UECM.from_ardl(ardl_model).fit()
    ru_specA_dummy_full[maturity] = uecm_fit
    print(f"\n{'='*70}\n  RU {maturity} (Full Sample + War Dummy, Spec A) — UECM Results\n{'='*70}")
    print(uecm_fit.summary())


  RU 1yr (Full Sample + War Dummy, Spec A) — UECM Results
                               UECM Model Results                              
Dep. Variable:        D.ru_spread_1y_m   No. Observations:                  164
Model:             UECM(1, 1, 1, 1, 1)   Log Likelihood                 -76.811
Method:                Conditional MLE   S.D. of innovations              1.784
Date:                 Wed, 02 Sep 2026   AIC                            175.622
Time:                         18:44:20   BIC                            209.654
Sample:                     02-01-2013   HQIC                           189.439
                          - 08-01-2026                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.2866      0.234     -1.223      0.223      -0.750       0.176
ru_spread_1y_m.L1      -0.3335      0

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

#### Note: Block 12 — Fair comparison, full sample + war dummy, Spec A

**Full results:**

| Country | Maturity | n | EC term (p) | war_dummy.L1 (p) | Verdict |
|---|---|---|---|---|---|
| KZ | 1yr | 82 | -0.200 (0.001) | -0.033 (0.925) | Cointegration |
| RU | 1yr | 164 | -0.334 (<0.001) | -0.074 (0.498) | Cointegration |
| RU | 5yr | 164 | -0.201 (<0.001) | -0.238 (0.075) | Cointegration |
| RU | 10yr | 164 | -0.164 (<0.001) | -0.256 (0.088) | Inconclusive |

**Correcting an earlier framing (Block 10):** with RU now tested on the
identical specification, the war dummy is statistically insignificant across
every model (all p ≥ 0.075) for both countries. This means the previously
reported KZ 1yr "cointegration with war dummy" result should NOT be
interpreted as evidence that the war produced a detectable level shift in the
long-run relationship — the dummy itself carries no independent significance.
Rather, both countries show a genuine long-run relationship between spread
and domestic-liquidity/oil/VIX across the full 2013(RU)/2019(KZ)-2026 sample,
largely independent of whether the war dummy is included.

**Implication for the paper's methodology:** this result should be framed as
evidence that a simple constant-shift dummy is not an adequate tool for
detecting the war's structural effect within the ARDL framework — motivating
the subsequent Markov-switching (Notebook 05, which detects regime changes
endogenously) and VAR/IRF (Notebook 06, which tests for changes in dynamic
transmission, not just level) analyses as necessary complements, rather than
redundant robustness checks.

**Cross-country comparison, corrected:** the pre-war-only results (Block 6)
remain the more informative direct test of H1's "similarity before the war"
claim: RU shows robust pre-war cointegration at all three maturities; KZ's
sample is too short to test this with confidence pre-war alone. The
full-sample results here should be read as a separate finding — a long-run
relationship exists across the full period for both countries — rather than
as confirmation of a war-driven shift specifically.

#### Note: Block 11 — Interpreting RU pre-war long-run coefficients

**Error-correction speed is stable across maturities** (-0.32 to -0.33,
p<0.001 at all three), indicating a consistent equilibrium-reversion
mechanism across the curve, with a half-life of roughly 1.8 months.

**RUONIA's long-run effect strengthens with maturity** (0.027 → 0.061 →
0.099, 1yr to 10yr), becoming both larger and more precisely estimated at
longer maturities. Interpreted as: persistent domestic liquidity conditions
are priced more fully into longer-duration credit risk, while short-term
rate movements (often transient) are less fully reflected in the 1yr spread.

**Oil and VIX show the opposite maturity pattern** — both significant at 1yr
(oil: 0.006, p=0.003; VIX: 0.020, p=0.006) but fading to insignificance by
5yr and 10yr. This suggests Russia's short-end spread is more exposed to
immediate global risk-sentiment shocks, while the long end is governed more
by domestic monetary conditions than by global risk factors.

**Oil's sign (positive) is notable and differs from Kazakhstan's expected
oil-exporter relationship** (negative, i.e., higher oil → lower risk).
Given oil's effect fades to insignificance at longer maturities, this is
best interpreted as a short-end-specific phenomenon — plausibly linked to
capital-flow or ruble-volatility dynamics coinciding with oil-price
movements in this sample — rather than a robust, curve-wide "Russia's credit
risk rises with oil prices" finding.

**Overall interpretation:** these results depict a Russian credit market
where short-term spreads are sentiment/global-shock driven, and long-term
spreads are more structurally anchored to domestic monetary conditions —
a coherent, maturity-differentiated transmission mechanism that provides a
clean pre-war baseline against which post-war changes (Notebooks 05-06) can
be compared.